# 🛡️ Aegis — Huấn luyện YOLO phát hiện ngã (Google Colab GPU)

Notebook này huấn luyện mô hình **YOLO11** phát hiện ngã trên bộ dữ liệu Kaggle `uttejkumarkandagatla/fall-detection-dataset` (bạn đã tải zip lên **Google Drive**), rồi xuất `best.pt` để cắm vào backend Aegis.

**3 lớp:** `0 = Fall Detected`, `1 = Walking`, `2 = Sitting` — ~485 ảnh, định dạng YOLO sẵn.

### Trước khi chạy
1. Menu **Runtime → Change runtime type → GPU (T4)** → Save.
2. **Runtime → Run all** (không cần Kaggle token — dữ liệu lấy từ Drive).
3. File Drive phải để chế độ **Anyone with the link** (Bất kỳ ai có liên kết) để `gdown` tải được.


## 1) Cài thư viện + kiểm tra GPU


In [ ]:
!pip -q install ultralytics gdown
import torch, ultralytics
print('Ultralytics:', ultralytics.__version__)
print('CUDA khả dụng:', torch.cuda.is_available(),
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (hãy bật GPU!)')


## 2) Tải dữ liệu từ Google Drive

Tải zip từ Drive bằng `gdown` rồi giải nén. Nếu `gdown` báo lỗi quota, dùng phương án mount Drive ở cuối cell (bỏ comment).


In [ ]:
import gdown, zipfile, os
FILE_ID = "1wzM-rjXffA89bKi1m5eUc6KusMd6taWF"   # ID file zip trên Google Drive của bạn
ZIP = "/content/dataset.zip"
gdown.download("https://drive.google.com/uc?id=" + FILE_ID, ZIP, quiet=False)

os.makedirs("/content/fall-detection-dataset", exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content/fall-detection-dataset")
print("\nĐã giải nén. Cây thư mục (rút gọn):")
!find /content/fall-detection-dataset -maxdepth 2 -type d | head -30

# --- Phương án dự phòng nếu gdown lỗi quota: mount Drive rồi copy ---
# from google.colab import drive; drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/ĐƯỜNG_DẪN_TỚI/dataset.zip' /content/dataset.zip


## 3) Chuẩn hoá thư mục + kiểm tra lớp

Tự dò thư mục gốc chứa `Images`, không phân biệt hoa/thường (`Images`/`images`, `Val`/`val`), rồi tạo chuẩn `images/{train,val}` + `labels/{train,val}` cho Ultralytics. In thử class id trong nhãn.


In [ ]:
import shutil, pathlib, glob, collections
search = pathlib.Path('/content/fall-detection-dataset')
img_parents = [p.parent for p in search.rglob('*') if p.is_dir() and p.name.lower() == 'images']
root = img_parents[0] if img_parents else search
print('Dataset root:', root)

def _src(kind, aliases):
    for kd in root.iterdir():
        if kd.is_dir() and kd.name.lower() == kind:
            for sd in kd.iterdir():
                if sd.is_dir() and sd.name.lower() in aliases:
                    return sd
    return None

plan = {
    ('images','train'): _src('images', {'train'}),
    ('images','val'):   _src('images', {'val','valid','validation'}),
    ('labels','train'): _src('labels', {'train'}),
    ('labels','val'):   _src('labels', {'val','valid','validation'}),
}
for (kind, split), s in plan.items():
    dst = root / kind / split
    dst.mkdir(parents=True, exist_ok=True)
    if s and s.resolve() != dst.resolve():
        shutil.copytree(s, dst, dirs_exist_ok=True)
    n = len(glob.glob(str(dst / '*')))
    print(f'{kind}/{split}: {n} files  (nguồn: {s})')

ids = collections.Counter()
for f in glob.glob(str(root / 'labels/train/*.txt')):
    for line in open(f):
        if line.strip(): ids[int(line.split()[0])] += 1
print('Phân bố class id trong nhãn train:', dict(sorted(ids.items())))
print('→ Kỳ vọng 0=Fall Detected, 1=Walking, 2=Sitting. Nếu khác, sửa names ở cell 4.')


## 4) Tạo `data.yaml`


In [ ]:
data_yaml = f'''\
path: {root}
train: images/train
val: images/val

nc: 3
names:
  0: Fall Detected
  1: Walking
  2: Sitting
'''
with open('/content/data.yaml','w') as f: f.write(data_yaml)
print(data_yaml)


## 5) Huấn luyện

`yolo11s.pt` cân bằng cho T4. Dữ liệu nhỏ nên bật augmentation + `patience=30` để tự dừng. Đổi `yolo11n.pt` nếu muốn nhẹ hơn.


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=-1,            # tự chọn batch vừa VRAM T4
    device=0,
    workers=2,
    project='/content/runs',
    name='fall_detection',
    patience=30,
    degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5, mosaic=1.0,
)
print('Weights tốt nhất:', '/content/runs/fall_detection/weights/best.pt')


## 6) Đánh giá & xem kết quả


In [ ]:
best = YOLO('/content/runs/fall_detection/weights/best.pt')
metrics = best.val(data='/content/data.yaml')
print(f'mAP50    : {metrics.box.map50:.3f}')
print(f'mAP50-95 : {metrics.box.map:.3f}')
from IPython.display import Image as IPImage, display
for p in ['results.png','confusion_matrix.png','val_batch0_pred.jpg']:
    fp = f'/content/runs/fall_detection/{p}'
    if os.path.exists(fp): display(IPImage(fp, width=720))


## 7) Suy luận thử trên vài ảnh val


In [ ]:
import glob
from IPython.display import Image as IPImage, display
test_imgs = glob.glob(str(root/'images/val/*'))[:4]
preds = best.predict(test_imgs, conf=0.25, save=True, project='/content/runs', name='test_preds', exist_ok=True)
for r in preds:
    names = [best.names[int(c)] for c in r.boxes.cls] if r.boxes is not None else []
    print(os.path.basename(r.path), '→', names)
for f in glob.glob('/content/runs/test_preds/*.jpg')[:4]:
    display(IPImage(f, width=480))


## 8) Xuất & tải `best.pt` về máy

Tải `best.pt` về, đặt vào `backend/models/fall_yolo.pt` trong dự án Aegis.


In [ ]:
best.export(format='onnx')   # tuỳ chọn: bản ONNX chạy nhanh trên CPU
from google.colab import files
files.download('/content/runs/fall_detection/weights/best.pt')


## 9) Tích hợp vào backend Aegis

1. Tạo `backend/models/` và chép `best.pt` vào, ví dụ `backend/models/fall_yolo.pt`.
2. Sửa `backend/.env`:
   ```
   MODEL_PATH=models/fall_yolo.pt
   ```
3. Khởi động lại backend (`uv run python main.py`).

Backend **tự nhận biết** đây là model phát hiện 3 lớp: gặp lớp `Fall Detected` là báo ngã, vẫn giữ bộ theo dõi thời gian để giảm báo nhầm. Để `MODEL_PATH=yolov8n-pose.pt` thì quay lại chế độ phân tích tư thế.
